# EDA — Encuesta
Lee el dataset transformado desde `staging/` (columnas ya normalizadas).

In [ ]:
import pandas as pd
from pathlib import Path

STAGING_DIR = Path('..') / 'staging'

# Toma el archivo staging más reciente (excluye el diccionario)
staging_files = sorted(
    [f for f in STAGING_DIR.glob('*.xlsx') if 'dictionary' not in f.name],
    key=lambda f: f.stat().st_mtime,
    reverse=True
)

if not staging_files:
    raise FileNotFoundError('No hay archivos en staging/. Corre primero pipeline.py')

latest = staging_files[0]
print(f'Leyendo: {latest.name}')

df = pd.read_excel(latest)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Strip a TODAS las columnas de texto del dataframe de una vez
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

print(f"Strip aplicado a {len(str_cols)} columnas.")

In [ ]:
# Crear ID correlativo tipo ENC-0001
df.insert(0, 'id', ['ENC-' + str(i).zfill(4) for i in range(1, len(df) + 1)])

print(df['id'].head())

In [ ]:
for col in df.columns:
    uniques = df[col].dropna().unique()
    print(f"\n{'='*60}")
    print(f"  {col}  ({len(uniques)} únicos)")
    print(f"{'='*60}")
    print(uniques)

## Explosión de columnas de opción múltiple → formato long

In [ ]:
# Categorías conocidas por columna (membership check, no split simple)
KNOWN_CATEGORIES = {
    '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar': [
        'Emprendimiento o Pequeña/Mediana empresa familiar',
        'Ventas y comercio de mercaderías',
        'Empleado público (del Estado, gobiernos locales o empresas públicas)',
        'Ama de casa / Cuidado del hogar',
        'Profesional independiente / Servicios profesionales',
        'Empleado privado',
        'Actividades agrícolas (producción, cultivos, procesamiento de productos del agro)',
        'Actividades acuícolas (camaroneras, piscinas de crianza de peces)',
        'Actividades ganaderas (crianza de pequeñas, medianas y grandes especies, procesamiento de cárnicos)',
        'Transporte (de mercaderías o bienes, buses, taxis, motoservicios, etc.)',
        'Jubilado o pensionista',
    ],
    '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado': [
        'Personas con discapacidad (de cualquier tipo)',
        'Adultos mayores (más de 65 años)',
        'Niños o niñas menores de 5 años',
        'Personas con enfermedades crónicas o catastróficas',
        'No Aplica',
    ],
}


def explode_multichoice(df: pd.DataFrame, col: str, categories: list) -> pd.DataFrame:
    rows = []
    for _, row in df[['id', col]].iterrows():
        raw = row[col]
        if pd.isna(raw) or str(raw).strip() == '':
            continue
        found = [cat for cat in categories if cat in str(raw)]
        remaining = str(raw)
        for cat in found:
            remaining = remaining.replace(cat, '')
        otros = [t.strip(' ,') for t in remaining.split(',') if t.strip(' ,')]
        for cat in found:
            rows.append({'id': row['id'], 'categoria': cat})
        if otros:
            rows.append({'id': row['id'], 'categoria': 'Otro'})
    return pd.DataFrame(rows)


COL_5 = '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar'
COL_8 = '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado'

df_long_actividad = explode_multichoice(df, COL_5, KNOWN_CATEGORIES[COL_5])
df_long_grupos    = explode_multichoice(df, COL_8, KNOWN_CATEGORIES[COL_8])

# Columna limpia en df original
def clean_original(val, categories):
    if pd.isna(val):
        return val
    found = [cat for cat in categories if cat in str(val)]
    return val if found else 'Otro'

df[COL_5 + '_clean'] = df[COL_5].apply(lambda x: clean_original(x, KNOWN_CATEGORIES[COL_5]))

print("── df_long_actividad ──")
print(df_long_actividad['categoria'].value_counts().to_string())
print("\n── df_long_grupos ──")
print(df_long_grupos['categoria'].value_counts().to_string())

## Normalización pregunta 11 — Experiencia de uso del seguro

In [ ]:
COL_11 = '11_experiencia_de_uso_del_seguro_en_la_atención_de_una_enfermedad_o_emergencia_colocar_las_ideas_centrales_y_palabras_claves_mencionadas_por_la_persona_encuestada_ej_atención_de_un_parto_buena_experiencia_con_el_seguro_cobertura_de_hospitalización_y_gastos_al_100_opinión_regular_sobre_la_clínica_utilizada_dificultad_en_llenar_formularios_de_reembolso_colocar_n_a_en_caso_que_la_persona_encuestada_no_haya_brindado_información_registrar_experiencias_tanto_positivas_como_negativas'

EXP_MAP = {
    # ── Positiva - Cirugía ──────────────────────────────────────────────────
    'AFILIADO COMENTA QUE TUVO UNA OPERACIÓN Y SU EXPERIENCIA FUE BUENA COMENTA QUE FUERON EFICIENTES.'
        : 'Positiva - Cirugía',
    'COMENTA QUE PARA UNA OPERACIÓN DE HIJA PARA VESÍCULA Y FUE SATISFACTORIA'
        : 'Positiva - Cirugía',
    'recientemente su hija tuvo una operación en la cual el seguro si le cubrio una gran parte de los gastos y la atención fue buena'
        : 'Positiva - Cirugía',

    # ── Positiva - Hospitalización ──────────────────────────────────────────
    'AFILIADO SATISFECHO CON LA ATENCIÓN RECIBIDA EN UNA HOSPITALIZACIÓN PASADA CON SU ESPOSA'
        : 'Positiva - Hospitalización',
    'El año pasado uno de sus hermanos acudio a un hospital particular pero no le atendieron, despues al usar la cobertura de inmedical recibio atencion hospitalaria y solo tuvieron que pagar la mitad de los gastos.'
        : 'Positiva - Hospitalización',
    'Infección Intestinal con hospitalización, fue buena la atención'
        : 'Positiva - Hospitalización',
    'Le gusta la cobertura que recibe ya que el titular está en un tratamiento que es muy efectivo, incluso este fin de semana tiene programado una cobertura hospitalaria, la cual se la entregan rapido y sin esperar.'
        : 'Positiva - Hospitalización',
    'LE GUSTA EL SEGURO PORQUE RECIENTEMENTE ESTUVO INTERNADA EN UNA CLINICA Y LOS GASTOS LO CUBRIÓ EL SEGURO AL IGUAL QUE LA MEDICACION'
        : 'Positiva - Hospitalización',
    'Hospitalización por emergencia por apendicitis; la atención fue buena.'
        : 'Positiva - Hospitalización',
    'Uno de los hijos recientemente tuvo una hospitalizacion y el seguo logró cubrir parte de los gastos médicos'
        : 'Positiva - Hospitalización',
    'El año pasado tuvo una hospitalizacion uno de los nietos del titular y si recibio una buena atencion y cobertura de parte del seguro'
        : 'Positiva - Hospitalización',
    'Estuvo interna por emergencia, comenta que la experiencia fue buena'
        : 'Positiva - Hospitalización',
    'MYU BUENA TENCION EN LA ATENCION DE 3 DIAS DE HOSPITALIZADA'
        : 'Positiva - Hospitalización',
    'BUENA ATENCION EN EL HOSPITAL'
        : 'Positiva - Hospitalización',

    # ── Positiva - Emergencia ───────────────────────────────────────────────
    'SU ESPOSO FUE ATENDIDO EN UNA EMERGENCIA Y QUEDO MUY SATISFECHA'
        : 'Positiva - Emergencia',
    'EN UNA EMERGENCIA QUE TUVO CON SU ESPOSO FUE BIEN ATENDIDA Y ESTA SATISFECHA'
        : 'Positiva - Emergencia',
    'Un sobrino recibió una atención por emergencia sin tener que esperar y le cubrio la totalidad de la atención y tambien los medicamentos'
        : 'Positiva - Emergencia',
    'FUE BIEN ATENDIDO EL PACIENTE EN EMERGENCIA'
        : 'Positiva - Emergencia',
    'BUENA EMERGENCIA'
        : 'Positiva - Emergencia',
    'MUY BUENA ATENCION EN EMERGENCIA'
        : 'Positiva - Emergencia',

    # ── Positiva - Parto / Maternidad ───────────────────────────────────────
    'TUVO UN PARTO Y UNA ATENCIÓN POR EMERGENCIA Y FUE BIEN ATENDIDO'
        : 'Positiva - Parto / Maternidad',
    'TUVO DOS EXPERIENCIAS DE PARTO EN LA PRIMERA SATISFECHO POR LA COBERTURA Y EN LA SEGUNDA DE LA MISMA MANERA'
        : 'Positiva - Parto / Maternidad',
    'Su esposa estuvo embarazada y pudo cubrir los gastos utilizando la cobertura y la atencion de el plan de salud'
        : 'Positiva - Parto / Maternidad',

    # ── Positiva - Odontología ──────────────────────────────────────────────
    'ESTA SATISFECHA CON LA ATENCIÓN EN ODONTOLOGÍA YA QUE ES LA ÚNICA QUE A USADO'
        : 'Positiva - Odontología',
    'Solo ha tenido citas en odontología y ha sido bueno el servicio'
        : 'Positiva - Odontología',

    # ── Positiva - Medicina general ─────────────────────────────────────────
    'UNICAMENTE FUE CON FIEBRE EN UNA OCACION Y FUE BIEN ATENDIDA'
        : 'Positiva - Medicina general',
    'Cuando siente algun malestar, utiliza la cobertura del seguro inmedical y si le ayudan con citas médicas'
        : 'Positiva - Medicina general',
    'ACUDIÓ POR UNA FIEBRE Y ESTA SATISFECHA'
        : 'Positiva - Medicina general',
    'recientemente ha tenido citas médicas y si le entregan el servicio de manera rapida y satisfactoria'
        : 'Positiva - Medicina general',

    # ── Positiva - General ──────────────────────────────────────────────────
    'Le gusta que el seguro ademas de las citas medicas tambien le da cobertura para realizar examenes médicos que son ordenados por el doctor'
        : 'Positiva - General',
    'Hasta ahora es muy buena la cobertura, le han dado atencion cuando lo necesitan y le dan cobertura para examenes médicos'
        : 'Positiva - General',

    # ── Negativa - Cobertura insuficiente ───────────────────────────────────
    'COMENTA QUE HABLANDO DE EXÁMENES NO TIENE COBERTURA PARA RX O LABORATORIO'
        : 'Negativa - Cobertura insuficiente',
    'NO CUBRIA EL SEGURO EL PROCEDIMIENTO QUE SE REALIZO'
        : 'Negativa - Cobertura insuficiente',
    'EL SEGURO NO CUBRIO NADA DE LA CIRUGIA DEL ESPOSO DE LA TITULAR'
        : 'Negativa - Cobertura insuficiente',
    'NO CUBRIO EL SEGURO LA OPERACION DE SU HIJO, NO LE REALIZARON EL REMBOLSO.'
        : 'Negativa - Cobertura insuficiente',
    'NO CUBRE EL MEDICAMENTO NI TIENEN TODAS LAS ESPECIALIDADES'
        : 'Negativa - Cobertura insuficiente',
    'NO CUBRE NADA EL SEGURO'
        : 'Negativa - Cobertura insuficiente',

    # ── Negativa - Acceso a especialidades ──────────────────────────────────
    'AFILIADA COMENTA QUE NO ESTA DE ACUERDO CON QUE TENGA QUE TENER DERIVACIÓN PARA ALUNA ESPECIALIDAD YA QUE LE TOMA TIEMPO RECIBIR LA SIGUIENTE CITA MEDICA.'
        : 'Negativa - Acceso a especialidades',
    'NO ATIENDEN ESPECIALISTAS, EL MISMO MEDICO ATIENDE A TODOS'
        : 'Negativa - Acceso a especialidades',
    'NO TIENE BUENA ATENCIÓN, YA QUE HASTA POR QUERER ODONTOLOGÍA DEBE PASAR POR MG'
        : 'Negativa - Acceso a especialidades',

    # ── Negativa - Atención deficiente ──────────────────────────────────────
    'AFILIADA COMENTA QUE A SOLICITADO CITAS MEDICAS Y NO LE HAN AGENDADO POR FALTA DE PAGO CON LOS CENTROS MÉDICOS QUE ESTÁN CERCANOS, USA POCO EL SEGURO DE INMEDICAL.'
        : 'Negativa - Atención deficiente',
    'ATENCIÓN EN MEDICINA GENERAL SOCIO CONSIDERA QUE FUE POCO SATISFACTORIA ADEMAS QUE LA MEDICINA FUE POCO CUBIERTA PARA LO QUE LE ENVIARON Y NECESITA'
        : 'Negativa - Atención deficiente',
    'OBTUVO UNA MALA EXPERIENCIA AL ACUDIR POR SU ESPOSO PERO NO FUE INTERNADO SITIO QUE NO FUE BIEN ATENDIDA TUVO QUE SOLICITAR QUE LE HICIERAN DEVOLUCIÓN  ECONÓMICA YA QUE LE TOCO SER ATENDIDO E OTRO LUGAR. A PARTE DE ESO SATISFECHA CON LA ATENCIÓN'
        : 'Negativa - Atención deficiente',
    'EN EMERGENCIA LA ATENCION NO FUE BUENA'
        : 'Negativa - Atención deficiente',

    # ── Mixta ────────────────────────────────────────────────────────────────
    'COMENTA QUE ACUDIÓ POR EMERGENCIA CON SIGNOS GRIPALES Y ESTUVO SATISFECHO CON LA COBERTURA DE LA MEDICINA Y LA ATENCIÓN.( COMENTA QUE EN LA FAMILIA TIENEN UNA ENFERMEDAD DE DIABETES TIPO 2 CON LA QUE NO CUENTAN CON LA COBERTURA EN MEDICAMENTOS Y DEMÁS COSAS QUE NECESITAN, POCO SATISFECHOS POR ESE LADO.'
        : 'Mixta',
    'NO TENIA COBERTURA PARA UNA RUPTURA DE TIBIA QUE SUFRIÓ SU HIJO, DE AHÍ COMENTA QUE LA ATENCIÓN ES BUENA'
        : 'Mixta',
    'AFILIADA DIO A LUZ CONTANDO CON EL SEGURO, COMENTA QUE FUE UNA EMERGENCIA QUE NO ESTUVO CUBIERTA EN SU TOTALIDAD Y TUVO QUE CUBRIR LA AFILIADA SIENDO DESPUÉS DEVUELTO POR EL SEGURO'
        : 'Mixta',
    'Le gusta la cobertura que brinda el seguro pero en especialidades no le gusta tener que esperar varios días para recibir una atencion en alguna especialidad'
        : 'Mixta',
    'SU ESPOSA FUE OPERADA Y ESTA SATISFECHO CON LA COBERTURA Y ATENCIÓN.. PERO TAMBIÉN EN CLÍNICA ROMERO COMENTA QUE NO LE GUSTO LA ATENCIÓN.'
        : 'Mixta',
    'BUENA EXPERIENCIA EN MG, MALA EXPERIENCIA EN ODONTOLOGIA Y MAL TRATO DEL MEDICO'
        : 'Mixta',
    'LA COBERTURA DEBERIA SER MAS AMPLIA EN MEDICAMENTOS, LA ATENCION ES BUENA'
        : 'Mixta',

    # ── Sin novedad ──────────────────────────────────────────────────────────
    'SIN NOVEDADES EN LA ATENCION' : 'Sin novedad',
    'SIN NOVEDADES'                : 'Sin novedad',
    'SIN NOVEDAD, TODO MUY BIEN'   : 'Sin novedad',
    'SIN NOVEDAD, BUENA ATENCION'  : 'Sin novedad',
}

POSITIVA_GENERAL_KEYWORDS = [
    'BUENA', 'BUEN', 'MUY BIEN', 'EXCELENTE', 'SATISFECH', 'BIEN ATEND',
    'NINGUNA EN PARTICULAR', 'NO TIENE QUEJAS', 'BUENAS EXPERIENCIAS',
    'BUENA INFORMACION', 'RECIBIO BUENOS', 'TELEMEDICA',
]

def categorize_exp(val):
    if pd.isna(val) or str(val).strip().upper() in ('N/A', 'NA', ''):
        return 'Sin novedad'
    v = str(val).strip()
    if v in EXP_MAP:
        return EXP_MAP[v]
    v_up = v.upper()
    if any(kw in v_up for kw in POSITIVA_GENERAL_KEYWORDS):
        return 'Positiva - General'
    return 'Sin clasificar'

df['11_experiencia_categoria'] = df[COL_11].apply(categorize_exp)

print(df['11_experiencia_categoria'].value_counts().to_string())
print(f"\nSin clasificar:")
print(df[df['11_experiencia_categoria'] == 'Sin clasificar'][COL_11].values)

In [ ]:
print(f"\nSin clasificar:")                                                                
print(df[df['11_experiencia_categoria'] == 'Sin clasificar'][COL_11].values)   

In [ ]:
df.columns.to_list()

## Normalización de zonas geográficas

In [ ]:
COL_ZONA = '3_ciudad_escribir_ej_quito'

# Paso 1: title case base
df[COL_ZONA] = df[COL_ZONA].str.strip().str.title()

# Paso 2: diccionario de correcciones (typos y variantes → valor canónico)
corrections = {
    'La Libertdad'                      : 'La Libertad',
    'La Ibertad'                        : 'La Libertad',
    'Santo Domingo De Los Colorados'    : 'Santo Domingo de los Colorados',
    'Santo Domingo De Los Tsachilas'    : 'Santo Domingo de los Colorados',
    'Tulcan'    : 'Tulcán',
    'Buena Fe'    : 'Buena Fé',
    'Duran'    : 'Durán',

}

df[COL_ZONA] = df[COL_ZONA].replace(corrections)

# Resultado
print(df[COL_ZONA].value_counts().to_string())


In [ ]:
MULTI_COLS = [
    '5_actividad_económica_de_la_persona_afiliada_indicar_la_o_las_principales_que_generan_ingresos_al_hogar',
    '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado',
]

for col in MULTI_COLS:
    print(f"\n{'='*70}")
    print(f"  {col}")
    print(f"{'='*70}")
    for val in df[col].dropna().unique():
        print(repr(val))

In [ ]:
COL_SN = '9_uso_del_plan_salud_red_en_el_último_año'

# Paso 1: title case base
df[COL_SN] = df[COL_SN].str.strip().str.title()

# Paso 2: diccionario de correcciones (typos y variantes → valor canónico)
corrections = {
    'SI'                      : 'Sí',
    'No'                        : 'No',
}

df[COL_SN] = df[COL_SN].replace(corrections)

# Resultado
print(df[COL_SN].value_counts().to_string())


In [ ]:
df_le = df.copy()

In [ ]:
col = '7_no_de_personas_aseguradas_en_el_núcleo_familiar_escribir_número_sin_espacios_ej_4'

df_le[col] = df_le[col].clip(upper=6)

In [ ]:
df_le[col].unique()  # debe devolver solo [1, 2, 3, 4, 5, 6]

In [ ]:
# ENC-0311: tenía "Adultos mayores (más de 65 años), No Aplica" en Q8
# → limpiar la columna raw en df_le Y eliminar la fila "No Aplica" del long
COL_8 = '8_grupos_de_atención_prioritaria_en_el_núcleo_asegurado'

df_le.loc[df_le['id'] == 'ENC-0311', COL_8] = 'Adultos mayores (más de 65 años)'

df_long_grupos = df_long_grupos[
    ~((df_long_grupos['id'] == 'ENC-0311') & (df_long_grupos['categoria'] == 'No Aplica'))
].reset_index(drop=True)

print("df_le Q8 para ENC-0311:", df_le.loc[df_le['id'] == 'ENC-0311', COL_8].values)
print(df_long_grupos[df_long_grupos['id'] == 'ENC-0311'])

In [ ]:
df_le['6_ingreso_promedio_mensual_tomando_como_referencia_500_usd'].unique()

In [ ]:
col = '6_ingreso_promedio_mensual_tomando_como_referencia_500_usd'

mapa_ingreso = {
    'Menos de 1 salario básico':      '1. Menos de 1 salario básico',
    'Al menos 1 salario básico':      '2. Al menos 1 salario básico',
    'Entre 1 y 2 salarios básicos':   '3. Entre 1 y 2 salarios básicos',
    'Más de 2 salarios básicos':      '4. Más de 2 salarios básicos',
    'Prefiere no responder':          '5. Prefiere no responder',
}

df_le[col] = df_le[col].map(mapa_ingreso)

In [ ]:
df_le['12_antes_de_tener_el_plan_de_salud_plan_salud_red_en_caso_de_enfermedad_o_emergencias_la_familia'].value_counts()

In [ ]:
col = '12_antes_de_tener_el_plan_de_salud_plan_salud_red_en_caso_de_enfermedad_o_emergencias_la_familia'

mapa_12 = {
    'IESS':                        'Intentó recibir (o recibió) atención en centro médico público',
    'SEGURO CAMPESINO':            'Intentó recibir (o recibió) atención en centro médico público',
    'USABA SEGURO ISSFA':          'Intentó recibir (o recibió) atención en centro médico público',
    'ANTES TENIAN SEGURO PRIVADO': 'Intentó recibir (o recibió) atención en centro médico público',
    'CLINICAS':                    'Pagaba de su bolsillo (ahorros o presupuesto familiar)',
    'PARTICULAR':                  'Pagaba de su bolsillo (ahorros o presupuesto familiar) ',
    'SE AUTOMEDICABA':             'Pagaba de su bolsillo (ahorros o presupuesto familiar) ',
    'NO MENCIONA':                 'Sin información',
}

df_le[col] = df_le[col].map(mapa_12).fillna(df_le[col])

In [ ]:
df_le['15_promedio_mensual_de_presupuesto_gastado_en_salud_en_el_núcleo_familiar'].unique()

In [ ]:
col = '15_promedio_mensual_de_presupuesto_gastado_en_salud_en_el_núcleo_familiar'

mapa_15 = {
    'Entre $10 y $50 dólares':   '1. Entre $10 y $50',
    'Entre $50 y $100 dólares':  '2. Entre $50 y $100',
    'Entre $100 y $200 dólares': '3. Entre $100 y $200',
    'Más de $200 dólares':       '4. Más de $200',
}

df_le[col] = df_le[col].map(mapa_15)

In [ ]:
df_le['9_uso_del_plan_salud_red_en_el_último_año'] = df_le['9_uso_del_plan_salud_red_en_el_último_año'].fillna('Si')

In [ ]:
df_le['7_no_de_personas_aseguradas_en_el_núcleo_familiar_escribir_número_sin_espacios_ej_4'].unique()


In [ ]:
df_le['9_uso_del_plan_salud_red_en_el_último_año'].value_counts()

In [ ]:
  # Diagnóstico de discrepancia "No aplica"                                                                                              
  cat = 'No Aplica'                                                                                                                                                                                                                                                               
  n_noaplica = (df_long_grupos['categoria'] == cat).sum()                                                                                
  total = len(df_le)

  print(f"n 'No Aplica' en df_long_grupos : {n_noaplica}")
  print(f"Total encuestados (len df_le)   : {total}")
  print(f"% crosstab                      : {n_noaplica/total*100:.4f}%")
  print()
  print(f"IDs con 'No Aplica':")
  print(df_long_grupos[df_long_grupos['categoria'] == cat]['id'].values)

In [ ]:
col_p10 = '10_necesidad_del_asegurado_a_en_acudir_a_chequeos_médicos'                                                                                                     
print(df_le[col_p10].value_counts(dropna=False))                                                                                       
print()                                                        
print("Nulos:", df_le[col_p10].isna().sum())                                                                                             
print("IDs con nulo:", df_le[df_le[col_p10].isna()]['id'].values)   

In [ ]:
print(df_le[df_le[col_p10].isna()]['id'].values) 

In [ ]:
import sys
sys.path.append('..')

from src.upload_to_sheets import upload_all

upload_all(df_le)

## Tablas de cruce (crosstabs)

In [ ]:
import importlib
import src.crosstabs as _ct
importlib.reload(_ct)
from src.crosstabs import generate_all

generate_all(df_le, df_long_actividad, df_long_grupos)